# HALO Engine: Reproducing Hardware Results
This notebook demonstrates the exact data pipeline used in *The HALO Engine: $\mathcal{O}(1)$-Step Compilation and Localized String Rupture for Lattice Gauge Theories on Quantum Hardware*. 

It provides three methods for verifying the manuscript's claims:
1. **Historical API Retrieval:** Downloading the exact raw counts generated during our IBM Quantum hardware runs (August 2026).
2. **Exact Analytical Reproduction:** Using classical statevector simulation to mathematically verify the $t=0.790$ topological crossing.
3. **Live QPU Execution:** Generating the hardware-ready circuits for ZNE, String Rupture, and Phase sweeps to execute on IBM Heron processors today.

### Dependencies
Ensure your environment matches the repository specifications.

In [ ]:
# Install dependencies from the repository requirements
!pip install -r ../requirements.txt

In [ ]:
# Import core modules
import json
import os
import numpy as np
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Ensure the HALO engine library is in the path
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from halo.hamiltonian import build_halo_hamiltonian
from halo.mitigation import build_folded_circuit


## Step 1: Historical QPU Retrieval (Primary Verification)
When we query the IBM API using our historical Job IDs, Qiskit does not re-execute the circuit. Instead, it downloads the exact physical measurement counts generated by the physical processors during our manuscript experiments. 

*Note: You must have an active IBM Quantum API token saved locally to run this cell.*

In [ ]:
# Initialize IBM Quantum Service
try:
    service = QiskitRuntimeService()
    print("Successfully connected to IBM Quantum Runtime.\n")
except Exception as e:
    print("IBM API Token not found. Please run QiskitRuntimeService.save_account(token='...')")

# Load the historical Job IDs map
data_path = os.path.join('..', 'data', 'halo_ibm_executions.json')
with open(data_path, "r") as f:
    job_map = json.load(f)

print("Scanning JSON and fetching ALL historical QPU executions from IBM Cloud...")
print("(This may take 1-2 minutes depending on API response times)\n")

# Recursive function to find all job ID arrays and fetch them
def verify_all_jobs(d, current_path=""):
    for key, value in d.items():
        if isinstance(value, dict):
            verify_all_jobs(value, current_path + key + " -> ")
        elif isinstance(value, list) and key.endswith('ids'):
            print(f"--- {current_path}{key} ({len(value)} Jobs) ---")
            
            for jid in value:
                try:
                    job = service.job(jid)
                    res = job.result()
                    data_bin = res[0].data
                    
                    # Handle EstimatorV2 Results
                    if hasattr(data_bin, 'evs'):
                        ev = np.atleast_1d(data_bin.evs)[0]
                        print(f"  [OK] {jid} (Estimator) | Expectation Value: {ev:.5f}")
                    
                    # Handle SamplerV2 Results
                    else:
                        try:
                            counts = data_bin.meas.get_counts()
                        except AttributeError:
                            counts = list(data_bin.values())[0].get_counts()
                        top_state = max(counts, key=counts.get)
                        top_prob = (counts[top_state] / sum(counts.values())) * 100
                        print(f"  [OK] {jid} (Sampler)   | Top State: |{top_state}⟩ ({top_prob:.1f}%)")
                        
                except Exception as e:
                    print(f"  [FAIL] {jid} | Error: {e}")
            print("") # Spacing between categories

# Execute the full verification sweep
verify_all_jobs(job_map["experiments"])
print("Historical Verification Complete.")

Successfully connected to IBM Quantum Runtime.

Scanning JSON and fetching ALL historical QPU executions from IBM Cloud...
(This may take 1-2 minutes depending on API response times)

--- coherence_wall_saturation -> 10_qubit_marrakesh -> job_ids (11 Jobs) ---
  [OK] d96ltot2su3c739h4170 (Estimator) | Expectation Value: -2.00059
  [OK] d96lpc8tcv6s73dju9tg (Estimator) | Expectation Value: -1.60320
  [OK] d96lo22f47jc73a5oha0 (Estimator) | Expectation Value: -2.90460
  [OK] d96ln752su3c739h3q90 (Estimator) | Expectation Value: -2.70685
  [OK] d96lmh8tcv6s73dju70g (Estimator) | Expectation Value: -2.05361
  [OK] d96llk2f47jc73a5oeq0 (Estimator) | Expectation Value: -3.62020
  [OK] d96lk3otcv6s73dju4i0 (Estimator) | Expectation Value: -1.69854
  [OK] d96ljb2f47jc73a5ocjg (Estimator) | Expectation Value: -2.95982
  [OK] d96lhbl2su3c739h3k8g (Estimator) | Expectation Value: -1.67713
  [OK] d96les52su3c739h3hm0 (Estimator) | Expectation Value: -2.21316
  [OK] d96lchd2su3c739h3f6g (Estimator)

## Step 2: Exact Analytical Reproduction (Statevector Math)
To prove the theoretical rigor of the Hamiltonian matrix without relying on historical IBM data, we can mathematically reproduce the exact localized string rupture crossover point (Figure 5) using classical Schrödinger dynamics.

In [14]:
# Recreate the exact 18.3% crossover at t = 0.790 analytically
N = 16
g_bare, m_bare = 1.012553, 0.506218
time_steps = np.linspace(0.78, 0.80, 200) # Micro-sweep around the crossover

print("Building 16-Qubit Hamiltonian Matrix...")
H_qlm = build_halo_hamiltonian(N, g_bare, m_bare, dt=1.0)
H_matrix = H_qlm.to_matrix(sparse=True)

# Staggered Vacuum & Stretched Meson State
qc_init = QuantumCircuit(N)
qc_init.x([3, 9, 15]) 
qc_init.x([3, 12])    
qc_init.x([4, 7, 10]) 

psi_0 = Statevector(qc_init).data
initial_bitstring = list(Statevector(qc_init).probabilities_dict().keys())[0]

survival_probs, breaking_probs = [], []

for t in time_steps:
    psi_t = spla.expm_multiply(-1j * t * H_matrix, psi_0)
    probs = Statevector(psi_t).probabilities_dict(decimals=6)
    
    survival_probs.append(probs.get(initial_bitstring, 0.0) * 100)
    
    broken_prob = sum(p for state, p in probs.items() if state[7] == '0' and state[8] == '0' and state != initial_bitstring)
    breaking_probs.append(broken_prob * 100)

surv_arr, brk_arr = np.array(survival_probs), np.array(breaking_probs)

# Mathematically locate the intersection
cross_indices = np.argwhere(np.diff(np.sign(surv_arr - brk_arr))).flatten()
if len(cross_indices) > 0:
    idx = cross_indices[0]
    t1, t2 = time_steps[idx], time_steps[idx+1]
    y1_s, y2_s = surv_arr[idx], surv_arr[idx+1]
    y1_b, y2_b = brk_arr[idx], brk_arr[idx+1]

    slope_s = (y2_s - y1_s) / (t2 - t1)
    slope_b = (y2_b - y1_b) / (t2 - t1)

    t_int = t1 + (y1_s - y1_b) / (slope_b - slope_s)
    p_int = y1_s + slope_s * (t_int - t1)

    print(f"\n[MATHEMATICAL PROOF] Localized String Rupture Point:")
    print(f"Calculated Time: t = {t_int:.6f} lattice units")
    print(f"Calculated Prob: P = {p_int:.6f}%")

Building 16-Qubit Hamiltonian Matrix...

[MATHEMATICAL PROOF] Localized String Rupture Point:
Calculated Time: t = 0.790325 lattice units
Calculated Prob: P = 18.314624%


## Step 3: Live Hardware Execution Pipelines
**⚠️ CRITICAL DISCLAIMER REGARDING HARDWARE DRIFT ⚠️**

Quantum hardware exhibits non-stationary noise. The $T_1$ and $T_2$ coherence times, as well as two-qubit gate infidelities, drift continuously. **Executing these circuits today will yield different unmitigated probability distributions than the historical data published in the manuscript.** 

The code blocks below generate the ISA-optimized circuits for the three core physical phenomena tracked in the paper. The actual `sampler.run()` commands are commented out to prevent accidental API credit consumption.

In [ ]:
# Setup for live execution
try:
    # Select the least busy Heron processor (target 16+ qubits)
    live_backend = service.least_busy(operational=True, simulator=False, min_num_qubits=16)
    pm = generate_preset_pass_manager(backend=live_backend, optimization_level=3)
    sampler = SamplerV2(mode=live_backend)
    sampler.options.default_shots = 2048
    print(f"Targeting QPU: {live_backend.name} for live compilation.")
except Exception as e:
    print("Hardware targeting failed. Ensure API token is valid.")

# ---------------------------------------------------------
# A. ZNE Unitary Folding Setup (10 Qubits, t=0.5)
# ---------------------------------------------------------
print("\nCompiling ZNE Unitary Folds...")
H_10q = build_halo_hamiltonian(10, g_coupling=1.012553, mass=0.506218, dt=1.0)
zne_circuits = []
for lambda_val in [1, 3, 5]:
    qc_fold = build_folded_circuit(H_10q, t_evo=0.5, reps=1, scale_factor=lambda_val, 
                                   num_qubits=10, initial_state_nodes=[3, 9, 0, 3, 2])
    zne_circuits.append(qc_fold)

isa_zne = pm.run(zne_circuits)
# Uncomment to execute:
# job_zne = sampler.run(isa_zne)
# print(f"ZNE Job ID: {job_zne.job_id()}")

# ---------------------------------------------------------
# B. Localized String Rupture Setup (16 Qubits, t=0.790)
# ---------------------------------------------------------
print("Compiling 16-Qubit String Rupture...")
H_16q = build_halo_hamiltonian(16, g_coupling=1.012553, mass=0.506218, dt=1.0)
qc_break = QuantumCircuit(16)
qc_break.x([3, 9, 15, 3, 12, 4, 7, 10]) # Combined vacuum, meson, and flux
qc_break.append(PauliEvolutionGate(H_16q, time=0.790, synthesis=LieTrotter(reps=1)), range(16))
qc_break.measure_all()

isa_break = pm.run([qc_break])
# Uncomment to execute:
# job_break = sampler.run(isa_break)
# print(f"String Rupture Job ID: {job_break.job_id()}")

# ---------------------------------------------------------
# C. Phase Diagram Parameter Sweep (16 Qubits, g=1.0 to 5.0)
# ---------------------------------------------------------
print("Compiling Phase Diagram Sweep (g-variance)...")
g_sweep_circuits = []
for g_val in [1.0, 3.0, 5.0]: # Sampled values to save execution time
    H_g = build_halo_hamiltonian(16, g_coupling=g_val, mass=0.506218, dt=1.0)
    qc_g = QuantumCircuit(16)
    qc_g.x([3, 9, 15, 3, 12, 4, 7, 10])
    qc_g.append(PauliEvolutionGate(H_g, time=0.8, synthesis=LieTrotter(reps=1)), range(16))
    qc_g.measure_all()
    g_sweep_circuits.append(qc_g)

isa_g_sweep = pm.run(g_sweep_circuits)
# Uncomment to execute:
# job_g_sweep = sampler.run(isa_g_sweep)
# print(f"G-Sweep Job ID: {job_g_sweep.job_id()}")

print("\nAll live execution blocks compiled and ready for dispatch.")

## Conclusion
This notebook verifies the core claims presented in the manuscript. By bridging exact classical statevector mathematics with historical QPU measurements, we demonstrate that the non-perturbative dynamics of Lattice Gauge Theories—specifically localized string rupture and dynamical phase transitions—can be empirically observed on current superconducting hardware. 

Crucially, the ability to extract these mesoscopic physical observables before succumbing to the $T_2$ coherence wall is exclusively enabled by the HALO Engine's $\mathcal{O}(1)$ constant-depth compilation architecture. The empirical logs fetched herein serve as the fundamental data provenance for the associated manuscript.